[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S12/S12_02_redes_recurrentes.ipynb)

# S12 · 02 · Redes recurrentes (RNN, LSTM, GRU) para predecir la ocupación

**Tiempo estimado:** 1 h · **Sprint 12** (UD 4.7) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. Convertir una serie en **ventanas deslizantes** (entrada = últimos 28 días, salida = próximos 7).
2. **Escalar sin fuga**: el escalador solo aprende del entrenamiento.
3. Construir y entrenar **SimpleRNN, LSTM y GRU** en Keras.
4. Compararlas con el **baseline** del notebook 01 (¿merece la pena una red?).

**Contexto.** Mismo problema que en el notebook 01 (previsión de ocupación a 7 días), ahora con redes que
"leen" la secuencia día a día y tienen memoria.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, random, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); keras.utils.set_random_seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")
print("TensorFlow", tf.__version__, "| filas:", len(ocup))

## 1. Idea: una red con memoria
Una red normal mira cada fila por separado. Una **red recurrente** lee la secuencia paso a paso y va
actualizando un "resumen" de lo visto (su **estado**).
- **SimpleRNN**: resumen simple; tiende a olvidar lo lejano.
- **LSTM**: añade *puertas* que deciden qué recordar, qué olvidar y qué sacar.
- **GRU**: versión más ligera de la LSTM (menos puertas, más rápida).

Para entrenarlas, la serie se corta en **ventanas**: cada ejemplo es un trozo de 28 días (entrada) y los
7 días siguientes (salida).

## 2. Variables de entrada y partición temporal
Cada día será un vector con: ocupación, festivo y el día de la semana y del año en forma de seno/coseno
(así "domingo" y "lunes" quedan cerca). **Todo es información conocida** hasta ese día.

In [ ]:
DIAS_TEST, VENTANA, HORIZONTE = 91, 28, 7

df = pd.DataFrame(index=ocup.index)
df["ocupacion"] = ocup["ocupacion_pct"]
df["festivo"] = ocup["festivo"]
dsem, danio = ocup.index.dayofweek.values, ocup.index.dayofyear.values
df["sem_sin"], df["sem_cos"] = np.sin(2 * np.pi * dsem / 7), np.cos(2 * np.pi * dsem / 7)
df["anio_sin"], df["anio_cos"] = np.sin(2 * np.pi * danio / 365.25), np.cos(2 * np.pi * danio / 365.25)

n_ent = len(df) - DIAS_TEST
print("Días de entrenamiento:", n_ent, "| días de prueba:", DIAS_TEST)

## 3. Escalado SIN fuga
Las redes aprenden mal con números grandes; se escala la ocupación a [0, 1].
**Regla de oro:** el escalador se ajusta (`fit`) **solo con el entrenamiento**; con la prueba solo se aplica (`transform`).
Si usáramos el máximo y el mínimo de toda la serie, estaríamos filtrando información del futuro.

**Ejercicio 1.** Crea `escalador = MinMaxScaler()`, ajústalo con la columna de ocupación de las primeras `n_ent` filas
y guarda en `ocup_esc` la ocupación de **toda** la serie transformada (array de una dimensión).

In [ ]:
# TODO: fit sobre df[["ocupacion"]].iloc[:n_ent]; transform sobre toda la columna; usa .ravel()
...
assert len(ocup_esc) == len(df), "ocup_esc debe tener una posición por día (transforma toda la serie)."
assert abs(ocup_esc[:n_ent].min()) < 1e-9 and abs(ocup_esc[:n_ent].max() - 1) < 1e-9, \
    "El escalador debe ajustarse solo con el entrenamiento: en entrenamiento el mínimo es 0 y el máximo 1."

In [ ]:
X_feat = df.copy()
X_feat["ocupacion"] = ocup_esc
matriz = X_feat.values.astype("float32")          # (días, 6 variables)
objetivo = ocup_esc.astype("float32")             # lo que queremos predecir
print("Matriz de variables:", matriz.shape)

## 4. Ventanas deslizantes
Para cada posición `t` (el "origen"): entrada = días `t-28 … t-1`; salida = días `t … t+6`.

**Ejercicio 2.** Completa `crear_ventanas(origenes)`, que recibe una lista de posiciones `t` y devuelve
`X` con forma `(n, 28, 6)` e `y` con forma `(n, 7)`.

In [ ]:
def crear_ventanas(origenes):
    # TODO: para cada t: matriz[t-VENTANA:t] y objetivo[t:t+HORIZONTE]; apila con np.array
    ...

# Orígenes de entrenamiento: todas las posiciones cuyo futuro (7 días) cae DENTRO del entrenamiento
orig_ent = list(range(VENTANA, n_ent - HORIZONTE + 1))
# Orígenes de prueba: igual que en el notebook 01, uno por semana
orig_test = [o for o in range(n_ent, len(df), HORIZONTE) if o + HORIZONTE <= len(df)]

X_ent, y_ent = crear_ventanas(orig_ent)
X_test, y_test = crear_ventanas(orig_test)
print("Entrenamiento:", X_ent.shape, y_ent.shape, "| Prueba:", X_test.shape, y_test.shape)
assert X_ent.shape[1:] == (VENTANA, 6) and y_ent.shape[1] == HORIZONTE, "X debe ser (n, 28, 6) e y (n, 7)."
assert np.allclose(X_ent[0, -1], matriz[VENTANA - 1]), "El último día de la ventana del origen t es t-1."

**Comprobación sin fuga:** ninguna ventana de entrenamiento debe tocar el periodo de prueba.

In [ ]:
assert max(orig_ent) + HORIZONTE - 1 < n_ent, "Fuga: una ventana de entrenamiento llega a días de prueba."
print("Sin fuga: el último día usado en entrenamiento es", df.index[max(orig_ent) + HORIZONTE - 1].date(),
      "y la prueba empieza el", df.index[n_ent].date())

## 5. Baseline en la misma escala de evaluación
El baseline estacional (repetir la última semana) sobre las mismas 13 semanas. Medimos el MAE en **puntos de ocupación**.

In [ ]:
def mae_puntos(y_esc, pred_esc):
    """Error absoluto medio en puntos de ocupación (deshaciendo el escalado)."""
    rango = escalador.data_max_[0] - escalador.data_min_[0]
    return float(np.mean(np.abs(y_esc - pred_esc)) * rango)

pred_base = np.array([objetivo[o - 7:o] for o in orig_test])       # última semana conocida
mae_base = mae_puntos(y_test, pred_base)
print(f"MAE baseline estacional: {mae_base:.2f} puntos")

## 6. Construir y entrenar las tres redes
La misma estructura, cambiando solo la capa recurrente. Modelos pequeños (32 unidades) y pocas épocas para que sea rápido.
Usamos como validación las **últimas 60 ventanas del entrenamiento** (sin barajar) y *early stopping*: se para cuando deja de mejorar.

In [ ]:
def construir(tipo):
    capa = {"SimpleRNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[tipo]
    modelo = keras.Sequential([
        layers.Input(shape=(VENTANA, 6)),
        capa(32),
        layers.Dense(HORIZONTE),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(0.005), loss="mae")
    return modelo

def entrenar(tipo, epocas=40):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(tipo)
    n_val = 60
    parada = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss")
    h = m.fit(X_ent[:-n_val], y_ent[:-n_val], validation_data=(X_ent[-n_val:], y_ent[-n_val:]),
              epochs=epocas, batch_size=32, shuffle=True, callbacks=[parada], verbose=0)
    return m, h

**Ejercicio 3.** Entrena las tres redes en un diccionario `modelos` con claves `"SimpleRNN"`, `"LSTM"` y `"GRU"`,
donde cada valor es la tupla `(modelo, historial)` que devuelve `entrenar`. Después se calcula el MAE de cada una en `resultados`.

In [ ]:
# TODO: recorre los tres nombres y llama a entrenar(nombre)
...
assert set(modelos) == {"SimpleRNN", "LSTM", "GRU"}, "modelos debe tener las claves SimpleRNN, LSTM y GRU."

resultados = {}
for nombre, (m, h) in modelos.items():
    pred = m.predict(X_test, verbose=0)
    resultados[nombre] = mae_puntos(y_test, pred)
    print(f"{nombre:10s} MAE prueba: {resultados[nombre]:.2f} puntos | épocas: {len(h.history['loss'])}")
print(f"{'Baseline':10s} MAE prueba: {mae_base:.2f} puntos")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (nombre, (m, h)) in zip(ejes, modelos.items()):
    ax.plot(h.history["loss"], label="entrenamiento"); ax.plot(h.history["val_loss"], label="validación")
    ax.set_title(nombre); ax.set_xlabel("Época")
ejes[0].set_ylabel("Error (MAE, escala 0-1)"); ejes[0].legend()
plt.tight_layout(); plt.show()

## 7. Comparar con el baseline y mirar una previsión
La pregunta importante no es "¿cuál red gana?" sino **"¿gana alguna al baseline?"**.

In [ ]:
mejor = min(resultados, key=resultados.get)
comp = pd.Series({**resultados, "Baseline estacional": mae_base}).sort_values()
print(comp.round(2).to_string())

m = modelos[mejor][0]
pred = m.predict(X_test, verbose=0)
rango = escalador.data_max_[0] - escalador.data_min_[0]
inv = lambda a: a * rango + escalador.data_min_[0]
fig, ax = plt.subplots()
real_cont = inv(y_test).ravel(); pred_cont = inv(pred).ravel(); base_cont = inv(pred_base).ravel()
ind = df.index[orig_test[0]:orig_test[0] + len(real_cont)]
ax.plot(ind, real_cont, color="black", label="Real"); ax.plot(ind, base_cont, alpha=.7, label="Baseline t-7")
ax.plot(ind, pred_cont, label=f"Red {mejor}")
ax.set_title("Previsión a 7 días (últimas 13 semanas)"); ax.set_ylabel("Ocupación (%)"); ax.legend()
plt.show()

## 8. Mini-reto integrador
Comprueba cuánto **empeora** una red si cometemos el error de escalar con toda la serie (fuga) o
si la ventana es demasiado corta. Entrena una **GRU** con ventana de 7 días (en lugar de 28) y guarda su MAE en `mae_ventana7`.
Pista: cambia `VENTANA` y regenera `matriz`-ventanas; recuerda restaurarlo al final para no romper el resto.
Guarda en `ventana_larga_mejor` (True/False) si con 28 días la GRU ha ido mejor que con 7.

In [ ]:
# TODO: guarda VENTANA, ponla a 7, rehace ventanas de entrenamiento/prueba con orígenes válidos (t >= 28 en ambos casos para comparar igual), entrena una GRU y restaura
...
print(f"GRU ventana 28: {resultados['GRU']:.2f} | GRU ventana 7: {mae_ventana7:.2f}")
assert VENTANA == 28, "Restaura VENTANA = 28 al terminar."
assert 0 < mae_ventana7 < 15, "mae_ventana7 debe ser un MAE en puntos de ocupación."
assert isinstance(ventana_larga_mejor, bool), "ventana_larga_mejor debe ser True o False."

## Preguntas de reflexión
1. ¿Por qué el escalador no debe ver los datos de prueba? Explícalo con un ejemplo de este notebook.
2. ¿Ha ganado alguna red al baseline? Si no, ¿qué le dirías al cliente antes de poner una red en producción?
3. Con solo ~1000 días de datos, ¿qué riesgo hay de que la red memorice (sobreajuste)? ¿Cómo lo vigila la validación?
4. ¿Qué tres decisiones te parecen más importantes (largo de ventana, unidades, épocas)? ¿Cómo las probarías sin usar la prueba?

## Resumen: qué has aprendido
- Una serie se convierte en ejemplos con **ventanas**; el escalado y la validación respetan el orden del tiempo.
- **RNN / LSTM / GRU** leen la secuencia con memoria; LSTM y GRU aguantan mejor las dependencias largas.
- Con pocos datos y un patrón claro, un **baseline** o un modelo estadístico pueden igualar a una red.
- Compara siempre con la misma partición y la misma métrica (MAE en puntos de ocupación).